In [18]:
### Cell Y1 – Compute IoU and Hausdorff Metrics

# install if needed
!pip install --quiet scikit-image scipy

import numpy as np
from skimage.metrics import hausdorff_distance
from scipy.spatial.distance import directed_hausdorff
from PIL import Image

# load & binarize reference
ref_img = Image.open(ref_path).convert("L")
ref_mask = (np.array(ref_img) > 128)  # bool mask

scores = []  # (path, iou, hausdorff)

for img_path in dataset.files:
    img = Image.open(img_path).convert("L")
    arr = np.array(img)
    mask = (arr > 128)

    # IoU
    inter = np.logical_and(ref_mask, mask).sum()
    union = np.logical_or(ref_mask, mask).sum()
    iou = inter / union if union>0 else 0

    # Hausdorff (symmetric)
    # skimage.hausdorff_distance needs two boolean arrays
    hd = hausdorff_distance(ref_mask, mask)

    scores.append((img_path, iou, hd))

# sort by IoU descending (most overlap first)
scores.sort(key=lambda x: x[1], reverse=True)

print("Top 5 by IoU then HD:", scores[:5])


Top 5 by IoU then HD: [('/content/shape2/original2.png', np.float64(1.0), np.float64(0.0)), ('/content/shape2/10386.png', np.float64(0.4699208443271768), np.float64(12.083045973594572)), ('/content/shape2/10452.png', np.float64(0.3989483310470965), np.float64(16.0)), ('/content/shape2/10659.png', np.float64(0.3808025177025964), np.float64(8.246211251235321)), ('/content/shape2/10731.png', np.float64(0.3575731497418244), np.float64(8.0))]


In [19]:
### Cell Z – Generate Self-Contained HTML with IoU & HD

import base64, os

html = [
    "<!DOCTYPE html>",
    "<html><head><meta charset='utf-8'><title>Shape Ranking</title></head><body>",
    "<h1>Ranking by IoU & Hausdorff</h1>",
    "<table style='width:100%; border-collapse: collapse;'>",
    "<tr><th>Image</th><th>IoU</th><th>Hausdorff</th></tr>"
]

for path, iou, hd in scores:
    ext  = path.rsplit(".",1)[-1].lower()
    mime = "image/png" if ext=="png" else "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html.append(
        "<tr style='border-bottom:1px solid #ccc;'>"
        f"<td><img src='data:{mime};base64,{b64}' width='200'></td>"
        f"<td style='text-align:center;'>{iou:.4f}</td>"
        f"<td style='text-align:center;'>{hd:.2f}</td>"
        "</tr>"
    )

html += ["</table></body></html>"]

out = "/content/results.html"
with open(out, "w") as f:
    f.write("\n".join(html))
print("Wrote ranking to", out)


Wrote ranking to /content/shape_ranking.html
